# Temperature

**Temperature** controls how predictable vs. creative Claude's output is. It's a value between 0 and 1 that reshapes the token-selection probabilities during generation:

1. **Tokenization** — your input is split into tokens
2. **Prediction** — the model computes a probability for each possible next token
3. **Sampling** — a token is chosen from that distribution

- **Low temperature (~0)** → nearly always picks the highest-probability token → deterministic, repetitive.
- **High temperature (~1)** → flattens the distribution → varied, creative, sometimes surprising.

| Range | Good for |
|-------|----------|
| 0.0 – 0.3 | Factual answers, coding, data extraction, moderation |
| 0.4 – 0.7 | Summarization, education, problem-solving |
| 0.8 – 1.0 | Brainstorming, creative writing, marketing, jokes |

> ⚠️ **Important deviation from our other notebooks.** The current flagship models — `claude-sonnet-5`, `claude-opus-4-8` — have **removed the `temperature` parameter**; sending it returns a 400 error. Anthropic now has those models manage sampling internally (you steer with prompting and the `effort` setting instead). To demonstrate temperature the way this lesson does, this notebook uses **`claude-haiku-4-5-20251001`**, which still accepts `temperature`.

## Setup + a `chat()` that accepts temperature

Same flexible `chat()` as the last lesson, now with a `temperature` parameter added to `params` (the course's exact change).

In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"   # Haiku 4.5 still supports the temperature param


def get_text(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return get_text(message)


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## Low temperature — run it a few times

At `temperature=0.0` the same prompt should give nearly the **same** movie idea every time.

In [2]:
prompt = "Give me a one-sentence idea for a movie. Just the idea, no preamble."

print("temperature = 0.0\n")
for i in range(3):
    messages = []
    add_user_message(messages, prompt)
    print(f"{i + 1}. {chat(messages, temperature=0.0)}\n")

temperature = 0.0

1. A grief-stricken woman discovers her late husband's secret second family and must decide whether to destroy their lives or find unexpected healing through them.

2. A grief-stricken woman discovers her late husband's secret second family and must decide whether to destroy their lives or find unexpected healing through them.

3. A grief-stricken woman discovers her late husband's secret second family and must decide whether to destroy their lives or find unexpected healing through connection with them.



## High temperature — run it a few times

At `temperature=1.0` the same prompt should give **different** ideas each run — different themes, characters, plots.

In [3]:
print("temperature = 1.0\n")
for i in range(3):
    messages = []
    add_user_message(messages, prompt)
    print(f"{i + 1}. {chat(messages, temperature=1.0)}\n")

temperature = 1.0

1. A grief-stricken woman discovers that her estranged daughter has been living a secret double life as a deep-cover spy, and they must work together on a dangerous mission to uncover who orchestrated their separation.

2. A con artist discovers the person they're trying to scam is actually their child from a past relationship they didn't know existed.

3. A cynical AI that was designed to predict human behavior begins to develop genuine emotions after years of analyzing people's most vulnerable moments.



## Key takeaway

Temperature changes the **probability** of varied output — it doesn't *guarantee* it. Even at 1.0 you may occasionally get similar responses, and even at 0.0 outputs aren't cryptographically identical. Match the setting to the job: low for consistent/factual, high for creative/brainstorming, medium for most general tasks.

**Modern footnote:** because the newest Claude models dropped `temperature`, the forward-looking way to get this behavior is prompting ("give me a wildly different idea each time") plus the `effort` parameter — but understanding temperature is still essential for working with most deployed models and other providers.